# ACE-Step 음악 생성 (Colab)

ACE-Step v1을 사용하여 텍스트 + 가사로 AI 음악을 생성합니다.

**요구사항**: GPU 런타임 (런타임 > 런타임 유형 변경 > T4 GPU)

| 항목 | 값 |
|------|----|
| 모델 | ACE-Step v1 (3.5B) |
| VRAM | ~8GB (T4 16GB에서 여유롭게 동작) |
| 생성 시간 | ~10-20초/분 (T4 기준) |

## 1. 설치

In [ ]:
!pip install -q ace-step soundfile IPython

## 2. GPU 확인

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("GPU가 없습니다! 런타임 > 런타임 유형 변경 > T4 GPU를 선택하세요.")

## 3. 모델 로드

첫 실행 시 HuggingFace에서 모델을 다운로드합니다 (~7GB, 약 2-3분 소요).

In [ ]:
from acestep.pipeline import ACEStepPipeline

pipeline = ACEStepPipeline.from_pretrained(
    "ACE-Step/ACE-Step-v1-3.5B",
    device_id=0,
    bf16=True,
    torch_compile=True,
    cpu_offload=True,
)
print("모델 로드 완료!")

## 4. 음악 생성 함수

In [ ]:
import soundfile as sf
from IPython.display import Audio, display


def generate_and_play(prompt, lyrics, duration=30, seed=42, filename="output.wav"):
    """음악을 생성하고 바로 재생합니다."""
    print(f"생성 중... (prompt: {prompt[:50]}...)")
    print(f"길이: {duration}초, seed: {seed}")

    audio = pipeline.text2music(
        prompt=prompt,
        lyrics=lyrics,
        duration=duration,
        infer_steps=27,
        guidance_scale=15.0,
        seed=seed,
    )

    sf.write(filename, audio.T, 44100)
    print(f"저장 완료: {filename}")
    display(Audio(filename, autoplay=True))
    return audio

## 5. 예제 1: K-Pop

In [ ]:
audio_kpop = generate_and_play(
    prompt="upbeat K-pop, catchy synth melody, female vocal, dance beat, 120 BPM",
    lyrics="""[verse]
빛나는 별처럼 우리 함께 달려가
멈추지 마 이 순간을 놓치지 마

[chorus]
Let's go, let's go, 더 높이 날아가
모두 함께 소리쳐 la la la la""",
    duration=30,
    seed=42,
    filename="kpop.wav",
)

## 6. 예제 2: Lo-fi 힙합 (기악)

In [ ]:
audio_lofi = generate_and_play(
    prompt="lo-fi hip hop, chill piano, vinyl crackle, jazzy chords, 85 BPM",
    lyrics="[instrumental]",
    duration=60,
    seed=123,
    filename="lofi.wav",
)

## 7. 예제 3: 록 발라드 (한국어 가사)

In [ ]:
audio_rock = generate_and_play(
    prompt="emotional rock ballad, electric guitar, powerful drums, male vocal, 75 BPM",
    lyrics="""[intro]
(guitar solo)

[verse]
어두운 밤을 걸어가며
너를 떠올려 다시 한번

[chorus]
잊지 못할 그 날의 기억
여전히 내 맘속에 남아있어

[outro]
(fade out)""",
    duration=45,
    seed=456,
    filename="rock_ballad.wav",
)

## 8. 직접 해보기

아래 셀에서 `prompt`와 `lyrics`를 바꿔서 나만의 음악을 만들어 보세요!

In [ ]:
# 직접 수정해 보세요!
my_audio = generate_and_play(
    prompt="여기에 스타일 태그를 입력하세요 (예: jazz, piano, soft vocal, 90 BPM)",
    lyrics="""[verse]
여기에 가사를 입력하세요

[chorus]
여기에 후렴을 입력하세요""",
    duration=30,
    seed=0,
    filename="my_song.wav",
)

## 9. 학습 포인트

### 프롬프트 구조
- **prompt**: 장르, 악기, 분위기, BPM 등 스타일 태그
- **lyrics**: `[verse]`, `[chorus]`, `[instrumental]` 등 구간 태그 + 가사
- 두 가지를 분리하여 스타일과 내용을 독립적으로 제어

### 품질 제어 파라미터
| 파라미터 | 설명 | 권장값 |
|---------|------|-------|
| `infer_steps` | 디노이징 스텝 수 (높을수록 고품질) | 27 (빠른) / 60 (고품질) |
| `guidance_scale` | 프롬프트 충실도 (높을수록 충실) | 15.0 |
| `seed` | 랜덤 시드 (동일 값 = 동일 결과) | 아무 정수 |
| `duration` | 생성 길이 (초) | 10~240 |

### Diffusion 기반 음악 생성 원리
이미지 생성의 Stable Diffusion과 동일한 원리:
1. 순수 노이즈에서 시작
2. 텍스트 조건에 맞게 점진적 디노이징
3. 오토인코더(DCAE)로 오디오 복원

### ACE-Step 고급 기능 (로컬 전용)
- **Retake**: 같은 설정에서 변형 생성
- **Repaint**: 특정 구간만 재생성
- **Edit**: 스타일/가사 변경하며 구조 유지
- **Extend**: 앞뒤로 자연스럽게 연장